In [1]:
import os, json
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

/home/mmk2266/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SRC = Path("data/rag_chunks_image_summary")      # input JSONs
DST = Path("data/rag_embeddings")                # output JSONLs
DST.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Alibaba-NLP/gte-large-en-v1.5"       # 1024-d, long context
BATCH = 64

model = SentenceTransformer(MODEL_ID, trust_remote_code=True)

In [3]:
def build_text(ch):
    md = ch.get("metadata", {})
    sec = md.get("section", "")
    page = md.get("page", "")
    head = f"[SECTION] {sec} [PAGE] {page}".strip()

    if ch.get("type") == "figure":
        cap  = ch.get("content", "") or ""
        summ = md.get("image_summary", "") or ""
        return f"{head}\n[FIGURE]\nCaption: {cap}\nVisual summary: {summ}".strip()
    else:  # paragraph (default)
        body = ch.get("content", "") or ""
        return f"{head}\n[PARAGRAPH]\n{body}".strip()

In [7]:
# def iter_chunks(path: Path):
#     data = json.loads(path.read_text())
#     return data.get("chunks", data)  # support list or {"chunks": [...]}

def iter_chunks(path: Path):
    # robust JSON load
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data                           # already a list of chunks
    if isinstance(data, dict):
        if "chunks" in data and isinstance(data["chunks"], list):
            return data["chunks"]             # {"chunks": [...]}
        # tolerate other wrappers like {"data":[...]}
        for k in ("data", "items"):
            if k in data and isinstance(data[k], list):
                return data[k]
        raise ValueError(f"Unexpected dict schema in {path}")
    raise ValueError(f"Unexpected JSON type {type(data)} in {path}")

In [8]:
def embed_texts(texts):
    # cosine-ready vectors (normalize=True); FAISS IndexFlatIP will work directly
    return model.encode(
        texts,
        batch_size=BATCH,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    ).astype("float32")

In [ ]:
for src in tqdm(sorted(SRC.glob("*.json"))):
    # collect
    entries, texts = [], []
    for ch in iter_chunks(src):
        md = ch.get("metadata", {})
        tfe = build_text(ch)
        entries.append({
            "id": ch.get("id"),
            "doc_id": src.stem.split(".")[0],
            "type": ch.get("type"),
            "page": md.get("page"),
            "section": md.get("section"),
            "text_for_embedding": tfe,
            "image_path": md.get("image_path") if ch.get("type") == "figure" else None,
            "image_summary": md.get("image_summary") if ch.get("type") == "figure" else None,
            "model": MODEL_ID,
        })
        texts.append(tfe)

    # embed
    embs = embed_texts(texts)

    # write JSONL (one file per source JSON)
    out_path = DST / f"{src.stem}.jsonl"
    with out_path.open("w") as f:
        for e, v in zip(entries, embs):
            e["embedding"] = [float(x) for x in v.tolist()]
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

print("Saved embeddings to:", DST)